# DS13 - Radix Sort using CUDA Python (Numba)

## Dataset Metadata

- Dataset Name: Sorting Benchmark Dataset
- Source: Kaggle, using the 10 Million Random Number Dataset
- Kaggle Dataset: https://www.kaggle.com/datasets/mehedihasand1497/10-million-random-number-dataset-for-ml
- Records: 10,000,000
- Data Type Used for Sorting: `uint32`
- Range: 0 to 4,294,967,295
- CUDA Operation: LSD Radix Sort

## CUDA Task

Input:
- Unsorted integer array generated from one numerical feature column.

Output:
- Sorted integer array.

Sorting Method:
- Least Significant Digit (LSD) Radix Sort.

Parallelization Strategy:
- Per-bit histogram/count
- Prefix offsets
- Stable scatter

Important for Kaggle:
- Add the 10 Million Random Number Dataset using Kaggle's "Add Data" panel.
- This notebook does not download the dataset.
- It only reads files from `/kaggle/input`.

The original dataset values are floating-point numbers between 0 and 1. They are converted to unsigned 32-bit integer keys using:

`key = round(value * (2^32 - 1))`

In [4]:
# Standard libraries used for paths and timing.
from pathlib import Path
import time

# Numerical, CSV, and GPU libraries available in Kaggle notebooks.
import numpy as np
import pandas as pd
from numba import cuda


# Kaggle automatically mounts added datasets here.
INPUT_ROOT = Path("/kaggle/input")

# Kaggle allows notebook outputs to be written here.
WORKING_ROOT = Path("/kaggle/working")

# Dataset metadata used in error messages and file discovery.
DATASET_URL = "https://www.kaggle.com/datasets/mehedihasand1497/10-million-random-number-dataset-for-ml"
DATASET_SLUG_HINT = "10-million-random-number-dataset-for-ml"


def find_csv_files(root=INPUT_ROOT):
    """Find CSV files from Kaggle input without downloading anything."""
    if not root.exists():
        raise FileNotFoundError(
            "Kaggle input directory was not found. Add the dataset in Kaggle: "
            f"{DATASET_URL}"
        )

    all_csv_files = sorted(root.rglob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
    preferred_files = [p for p in all_csv_files if DATASET_SLUG_HINT in str(p).lower()]
    csv_files = preferred_files or all_csv_files

    if not csv_files:
        raise FileNotFoundError(
            "No CSV file was found under /kaggle/input. Add the Kaggle dataset to this notebook."
        )

    print("CSV files detected:")
    for path in csv_files:
        print(f"  {path} ({path.stat().st_size / (1024 ** 3):.2f} GB)")
    return csv_files


def columns_look_like_data(columns):
    """Detect headerless CSV files where the first row was read as column names."""
    labels = pd.Series([str(c) for c in columns])
    parsed = pd.to_numeric(labels, errors="coerce")
    return parsed.notna().mean() > 0.80


def detect_numeric_columns(csv_path, feature_limit=50):
    """Return pandas read options and numeric feature columns."""
    preview = pd.read_csv(csv_path, nrows=256)
    read_kwargs = {}

    # If column labels look numeric, the file probably has no header.
    if columns_look_like_data(preview.columns):
        read_kwargs = {"header": None}
        preview = pd.read_csv(csv_path, nrows=256, header=None)

    numeric_columns = []
    for col in preview.columns:
        numeric = pd.to_numeric(preview[col], errors="coerce")
        if numeric.notna().mean() > 0.95:
            numeric_columns.append(col)

    if not numeric_columns:
        raise ValueError("Could not detect numeric feature columns in the Kaggle CSV.")

    return read_kwargs, numeric_columns[:feature_limit]


def select_dataset_csv():
    """Use the largest CSV from the Kaggle-mounted dataset folder."""
    csv_path = find_csv_files()[0]
    print(f"Using CSV: {csv_path}")
    return csv_path


def dataframe_to_float32(df):
    """Convert a pandas chunk to contiguous float32 values for GPU transfer."""
    arr = df.apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32, copy=True)
    if np.isnan(arr).any():
        arr = np.nan_to_num(arr, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return np.ascontiguousarray(arr)


# Stop early if the Kaggle notebook is not using a GPU.
if not cuda.is_available():
    raise RuntimeError("CUDA is not available. In Kaggle, enable Settings -> Accelerator -> GPU.")

device = cuda.get_current_device()
print(f"CUDA device: {device.name.decode() if isinstance(device.name, bytes) else device.name}")
WORKING_ROOT.mkdir(parents=True, exist_ok=True)

CUDA device: Tesla T4


In [5]:
from numba import cuda, int32


# Number of CUDA threads per block.
TPB = 256

# The dataset has 50 numerical features; one feature is used as the sort input.
FEATURE_LIMIT = 50
SELECTED_FEATURE_INDEX = 0

# Read up to 10 million values from the selected feature.
ROW_LIMIT = 10_000_000

# Full NumPy verification is slower and uses more memory, so it is optional.
VERIFY_WITH_NUMPY = False

# Saving 10M uint32 values is about 40 MB, acceptable for Kaggle output if needed.
SAVE_FULL_SORTED_ARRAY = True


@cuda.jit
def count_zeros_kernel(keys, zero_counts, n, bit):
    """Count zero bits for one radix pass in each CUDA block."""
    smem = cuda.shared.array(shape=TPB, dtype=int32)
    tid = cuda.threadIdx.x
    i = cuda.blockIdx.x * cuda.blockDim.x + tid

    is_zero = 0
    if i < n:
        is_zero = 1 if ((keys[i] >> bit) & 1) == 0 else 0

    smem[tid] = is_zero
    cuda.syncthreads()

    # Block-level reduction creates one zero count per block.
    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tid < stride:
            smem[tid] += smem[tid + stride]
        cuda.syncthreads()
        stride //= 2

    if tid == 0:
        zero_counts[cuda.blockIdx.x] = smem[0]


@cuda.jit
def scatter_bit_kernel(keys_in, keys_out, zero_offsets, one_offsets, n, bit):
    """Stable scatter for one bit: all 0-bit keys first, then all 1-bit keys."""
    smem = cuda.shared.array(shape=TPB, dtype=int32)
    tid = cuda.threadIdx.x
    block_start = cuda.blockIdx.x * cuda.blockDim.x
    i = block_start + tid

    is_valid = i < n
    is_zero = 0
    if is_valid:
        is_zero = 1 if ((keys_in[i] >> bit) & 1) == 0 else 0

    smem[tid] = is_zero
    cuda.syncthreads()

    # Inclusive prefix sum of zero flags inside one block.
    offset = 1
    while offset < cuda.blockDim.x:
        add_value = 0
        if tid >= offset:
            add_value = smem[tid - offset]
        cuda.syncthreads()
        smem[tid] += add_value
        cuda.syncthreads()
        offset *= 2

    if is_valid:
        zeros_through_current = smem[tid]

        if is_zero == 1:
            # Rank among all keys with bit value 0.
            position = zero_offsets[cuda.blockIdx.x] + zeros_through_current - 1
        else:
            # Rank among all keys with bit value 1.
            ones_before_current = tid - zeros_through_current
            position = one_offsets[cuda.blockIdx.x] + ones_before_current

        keys_out[position] = keys_in[i]


def build_bit_offsets(zero_counts, n):
    """Build per-block prefix offsets from block zero counts."""
    num_blocks = zero_counts.size

    block_sizes = np.full(num_blocks, TPB, dtype=np.int32)
    block_sizes[-1] = n - (num_blocks - 1) * TPB

    zero_counts = zero_counts.astype(np.int32, copy=False)
    one_counts = block_sizes - zero_counts

    zero_offsets = np.empty(num_blocks, dtype=np.int32)
    one_offsets = np.empty(num_blocks, dtype=np.int32)

    total_zeros = int(zero_counts.sum())
    zero_offsets[0] = 0
    one_offsets[0] = total_zeros

    if num_blocks > 1:
        zero_offsets[1:] = np.cumsum(zero_counts[:-1], dtype=np.int64).astype(np.int32)
        one_offsets[1:] = (
            total_zeros + np.cumsum(one_counts[:-1], dtype=np.int64)
        ).astype(np.int32)

    return zero_offsets, one_offsets


def cuda_radix_sort_uint32(keys):
    """LSD radix sort for unsigned 32-bit integer keys."""
    keys = np.ascontiguousarray(keys, dtype=np.uint32)
    n = keys.size
    if n <= 1:
        return keys.copy()

    num_blocks = (n + TPB - 1) // TPB
    d_in = cuda.to_device(keys)
    d_out = cuda.device_array_like(d_in)
    d_zero_counts = cuda.device_array(num_blocks, dtype=np.int32)

    start = time.perf_counter()

    # Process bits from least significant to most significant.
    for bit in range(32):
        count_zeros_kernel[num_blocks, TPB](d_in, d_zero_counts, n, bit)

        # Copy compact block counts to CPU for prefix offsets.
        zero_counts = d_zero_counts.copy_to_host()
        zero_offsets, one_offsets = build_bit_offsets(zero_counts, n)

        d_zero_offsets = cuda.to_device(zero_offsets)
        d_one_offsets = cuda.to_device(one_offsets)
        scatter_bit_kernel[num_blocks, TPB](d_in, d_out, d_zero_offsets, d_one_offsets, n, bit)

        # Swap buffers for the next bit pass.
        d_in, d_out = d_out, d_in

        if bit in (0, 7, 15, 23, 31):
            cuda.synchronize()
            print(f"Completed radix pass for bit {bit:>2}")

    cuda.synchronize()
    elapsed = time.perf_counter() - start
    return d_in.copy_to_host(), elapsed


def float_values_to_uint32_keys(values):
    """Convert dataset float values in [0, 1] to uint32 sorting keys."""
    clean = np.nan_to_num(values.astype(np.float32, copy=False), nan=0.0, posinf=1.0, neginf=0.0)
    clipped = np.clip(clean, 0.0, 1.0)
    max_key = np.float64(np.iinfo(np.uint32).max)
    return np.rint(clipped.astype(np.float64) * max_key).astype(np.uint32)

In [6]:
# Select the Kaggle-mounted CSV dataset.
csv_path = select_dataset_csv()
read_kwargs, numeric_columns = detect_numeric_columns(csv_path, feature_limit=FEATURE_LIMIT)

if SELECTED_FEATURE_INDEX >= len(numeric_columns):
    raise IndexError(f"SELECTED_FEATURE_INDEX must be less than {len(numeric_columns)}")

selected_column = numeric_columns[SELECTED_FEATURE_INDEX]
print(f"Selected feature column: {selected_column}")

# Read one feature column, producing up to 10,000,000 floating-point values.
df = pd.read_csv(
    csv_path,
    usecols=[selected_column],
    nrows=ROW_LIMIT,
    **read_kwargs,
)

values = pd.to_numeric(df.iloc[:, 0], errors="coerce").fillna(0.0).to_numpy(dtype=np.float32)
keys = float_values_to_uint32_keys(values)

print(f"Loaded values: {values.size:,}")
print(f"Converted key dtype: {keys.dtype}")
print(f"Unsorted key range: min={keys.min()}, max={keys.max()}")

# Run CUDA LSD radix sort.
sorted_keys, elapsed = cuda_radix_sort_uint32(keys)

# Monotonic check verifies final sorted order.
is_sorted = bool(np.all(sorted_keys[:-1] <= sorted_keys[1:]))

print("\nDS13 Radix Sort complete.")
print(f"Keys sorted: {sorted_keys.size:,}")
print(f"Sorted correctly by monotonic check: {is_sorted}")
print(f"Elapsed time: {elapsed:.2f} seconds")
print("First 10 sorted keys:", sorted_keys[:10])
print("Last 10 sorted keys:", sorted_keys[-10:])

if VERIFY_WITH_NUMPY:
    # Optional full CPU verification.
    start_np = time.perf_counter()
    numpy_sorted = np.sort(keys)
    np_elapsed = time.perf_counter() - start_np
    print(f"Matches NumPy sort: {np.array_equal(sorted_keys, numpy_sorted)}")
    print(f"NumPy sort elapsed time: {np_elapsed:.2f} seconds")

# Save a small CSV preview for easy inspection in Kaggle output.
sample_path = WORKING_ROOT / "ds13_radix_sort_sorted_sample.csv"
pd.DataFrame({"sorted_uint32_key": sorted_keys[:1000]}).to_csv(sample_path, index=False)
print(f"Sample sorted keys saved to: {sample_path}")

if SAVE_FULL_SORTED_ARRAY:
    npy_path = WORKING_ROOT / "ds13_radix_sort_sorted_uint32.npy"
    np.save(npy_path, sorted_keys)
    print(f"Full sorted uint32 array saved to: {npy_path}")

CSV files detected:
  /kaggle/input/datasets/mehedihasand1497/10-million-random-number-dataset-for-ml/random_numbers.csv (4.95 GB)
Using CSV: /kaggle/input/datasets/mehedihasand1497/10-million-random-number-dataset-for-ml/random_numbers.csv
Selected feature column: feature_1
Loaded values: 10,000,000
Converted key dtype: uint32
Unsorted key range: min=636, max=4294967295
Completed radix pass for bit  0
Completed radix pass for bit  7
Completed radix pass for bit 15
Completed radix pass for bit 23
Completed radix pass for bit 31

DS13 Radix Sort complete.
Keys sorted: 10,000,000
Sorted correctly by monotonic check: True
Elapsed time: 0.41 seconds
First 10 sorted keys: [ 636  685  811 1162 1312 1551 2019 2591 3115 3301]
Last 10 sorted keys: [4294961407 4294962943 4294963455 4294963711 4294963711 4294965247
 4294965503 4294965503 4294966783 4294967295]
Sample sorted keys saved to: /kaggle/working/ds13_radix_sort_sorted_sample.csv
Full sorted uint32 array saved to: /kaggle/working/ds13_rad